In [5]:
"""
Extrae la serie de humedad de suelo relativa quincenal para un punto,
usando Sentinel-1 GRD (radar SAR) con el método de change detection
(Bauer-Marschallinger et al., 2019 - TU Wien).

MÉTODO: el backscatter de radar (VV, en dB) depende de la humedad del
suelo, pero también de la geometría de observación y la vegetación. El
change detection normaliza esto comparando cada valor contra el rango
histórico observado en ese MISMO punto:

    SM_index = (VV_actual - VV_seco) / (VV_húmedo - VV_seco) * 100

donde VV_seco y VV_húmedo son los percentiles 5 y 95 (no el mínimo/
máximo exacto, para ser robusto a ruido speckle) de toda la serie
histórica en ese punto.

CÓMO LEER EL RESULTADO
-----------------------
sm_index    Índice de humedad de suelo relativa (0-100%). NO es
            humedad volumétrica absoluta (m3/m3) - es una posición
            relativa entre el estado más seco y más húmedo observado
            históricamente en ESE punto específico. Sirve para comparar
            la misma parcela en el tiempo, no para comparar valores
            absolutos entre parcelas distintas con historiales
            distintos.
n_imagenes  Cuántas escenas de Sentinel-1 promedia esa quincena (para
            saber si el dato viene de 1 sola pasada o de varias).

LIMITACIONES A TENER EN CUENTA:
- Órbita: se usa una sola dirección de órbita (ASCENDING por defecto)
  para todo el análisis. Mezclar ascendente/descendente introduce
  diferencias de ángulo de incidencia que contaminan la comparación
  temporal - el objetivo del análisis pesa más que tener más imágenes.
- Vegetación densa: en dosel muy cerrado (selva, cultivo maduro), el
  radar "ve" más la vegetación que el suelo, y la relación
  backscatter-humedad se debilita. Es más confiable en suelo desnudo o
  vegetación baja/dispersa.
- Frecuencia: la revisita de Sentinel-1 es de 6-12 días según época y
  disponibilidad de satélites (se degradó a ~12 días desde que S1B
  dejó de operar en dic-2021) -> algunas quincenas pueden tener 0, 1 o
  2 imágenes. Se deja como NaN cuando no hay ninguna.
- Es un índice relativo, calibrado con el propio historial del punto
  -> los primeros años de la serie ayudan a "aprender" el rango
  seco/húmedo; cuanto más corta la serie histórica disponible, menos
  confiable el rango de referencia.

CITA: Bauer-Marschallinger, B., et al. (2019). Toward global soil
moisture monitoring with Sentinel-1: Harnessing assets and overcoming
obstacles. IEEE TGRS, 57(1), 520-539.
"""
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee

from test_period_utils import build_biweekly_periods


def get_soil_moisture_biweekly(lat, lon, start_date="2016-01-01", end_date=None,
                                orbit_pass="ASCENDING"):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy
    orbit_pass: "ASCENDING" o "DESCENDING" - se usa una sola dirección
                para toda la serie, por consistencia geométrica
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    # Verificamos ANTES de procesar cuál dirección de órbita tiene más
    # cobertura para este punto específico -> la disponibilidad de
    # Sentinel-1 varía mucho por región y no se puede asumir de antemano.
    base_coll = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(point)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filterDate(str(start), str(end + timedelta(days=1)))
    )
    n_asc = base_coll.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')).size().getInfo()
    n_desc = base_coll.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')).size().getInfo()
    print(f"[DEBUG] cobertura disponible: {n_asc} imágenes ASCENDING, {n_desc} imágenes DESCENDING")

    if orbit_pass == "ASCENDING" and n_asc < n_desc:
        print(f"[AVISO] ASCENDING tiene menos cobertura que DESCENDING para este punto "
              f"-> cambiando automáticamente a DESCENDING. Pasá orbit_pass explícito "
              f"si preferís forzar una dirección igual.")
        orbit_pass = "DESCENDING"
    elif orbit_pass == "DESCENDING" and n_desc < n_asc:
        print(f"[AVISO] DESCENDING tiene menos cobertura que ASCENDING para este punto "
              f"-> cambiando automáticamente a ASCENDING. Pasá orbit_pass explícito "
              f"si preferís forzar una dirección igual.")
        orbit_pass = "ASCENDING"

    s1_coll = base_coll.filter(ee.Filter.eq('orbitProperties_pass', orbit_pass)).select('VV')

    # Referencia seco/húmedo: percentiles 5 y 95 de TODA la serie histórica
    # en este punto (más robusto a ruido speckle que el min/max exacto)
    ref_stats = s1_coll.reduce(ee.Reducer.percentile([5, 95])).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=10,  # resolución nativa de Sentinel-1 GRD
        maxPixels=1e9
    )
    vv_seco = ee.Number(ref_stats.get('VV_p5'))
    vv_humedo = ee.Number(ref_stats.get('VV_p95'))
    rango = vv_humedo.subtract(vv_seco)

    print(f"[DEBUG] referencia histórica: VV seco (p5)={ref_stats.get('VV_p5').getInfo():.2f} dB, "
          f"VV húmedo (p95)={ref_stats.get('VV_p95').getInfo():.2f} dB")

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered = s1_coll.filterDate(p_start, p_end)
        n_imagenes = filtered.size()

        mean_vv = ee.Image(ee.Algorithms.If(
            n_imagenes.gt(0),
            filtered.mean().rename('VV'),
            ee.Image.constant(0).rename('VV').selfMask()
        ))

        # sm_index se calcula como banda de imagen (no como valor suelto con
        # ee.Algorithms.If) -> hereda la máscara de mean_vv automáticamente,
        # y reduceRegion se encarga de convertir "sin dato" en null de forma
        # natural, sin pasar por .set() con un None explícito (Dictionary.set
        # del cliente Python no acepta null como value).
        sm_index_img = (
            mean_vv.subtract(vv_seco).divide(rango).multiply(100)
            .max(0).min(100).rename('sm_index')
        )
        # n_imagenes como banda separada (constant, sin máscara) -> siempre
        # se reporta el conteo real, incluso cuando sm_index sale null
        n_img = ee.Image.constant(n_imagenes).toInt().rename('n_imagenes')

        combined = sm_index_img.addBands(n_img)

        stats = combined.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=10,
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para todos los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            'lat': lat,
            'lon': lon,
            'sm_index': round(props.get('sm_index'), 2) if props.get('sm_index') is not None else None,
            'n_imagenes': props.get('n_imagenes'),
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    filas_nulas = df['sm_index'].isna().sum()
    if filas_nulas > 0:
        primeras_nulas = df[df['sm_index'].isna()]['label'].tolist()
        print(f"[AVISO] {filas_nulas} quincena(s) sin ninguna imagen de Sentinel-1 "
              f"disponible: {primeras_nulas}")

    return df


def save_soil_moisture_profile(df, out_prefix="soil_moisture_biweekly", output_dir="../databases"):
    """
    Guarda la serie de humedad de suelo con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Sugarcane_QLD
    LAT = 7.4584221918243045
    LON = -73.222052853104
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    df = get_soil_moisture_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_soil_moisture_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-13
[DEBUG] cobertura disponible: 318 imágenes ASCENDING, 323 imágenes DESCENDING
[AVISO] ASCENDING tiene menos cobertura que DESCENDING para este punto -> cambiando automáticamente a DESCENDING. Pasá orbit_pass explícito si preferís forzar una dirección igual.
[DEBUG] referencia histórica: VV seco (p5)=-10.46 dB, VV húmedo (p95)=-3.71 dB
[AVISO] 26 quincena(s) sin ninguna imagen de Sentinel-1 disponible: ['2016-01_Q1', '2016-02_Q2', '2016-03_Q2', '2016-05_Q1', '2016-06_Q2', '2016-08_Q1', '2016-09_Q1', '2016-11_Q2', '2017-01_Q1', '2017-03_Q2', '2018-06_Q2', '2018-07_Q1', '2018-07_Q2', '2018-08_Q1', '2018-08_Q2', '2018-09_Q1', '2018-09_Q2', '2018-10_Q1', '2019-04_Q1', '2021-02_Q1', '2021-12_Q2', '2022-01_Q1', '2022-01_Q2', '2022-02_Q1', '2022-02_Q2', '2024-08_Q1']
CSV guardado en ../databases/soil_moisture_biweekly-v260813180447.csv (254x7)
  periodo_inicio periodo_fin       label       lat        lon  sm_index  \
0     201

ASCENDING tiene menos cobertura que DESCENDING para este punto.

Definitivamente no es igual en todo el mundo, y hay tres razones distintas que se combinan:

1. La cobertura es planificada por región, no uniforme por diseño. ESA usa "Observation Scenarios" predefinidos para programar las adquisiciones globalmente — es decir, alguien en ESA literalmente decide de antemano qué zonas del planeta se fotografían con qué frecuencia, en qué modo, y en qué dirección de órbita. A diferencia de misiones anteriores (ERS, Envisat) donde la planificación se basaba en pedidos de usuarios, Sentinel se opera con una planificación predefinida, priorizando a los países miembro de la UE y los servicios de Copernicus. Europa recibe cobertura mucho más densa que, por ejemplo, el interior rural de Queensland.

2. La frecuencia de revisita varía naturalmente con la latitud. La tasa de revisita (ascendente/descendente) es significativamente mayor en latitudes altas que en el ecuador: menos de 1 día en el Ártico, 1-3 días en las principales rutas de navegación, y 3 días en el ecuador — es geometría orbital pura, nada que ver con decisiones de planificación. Tanto tus coordenadas de Colombia (cerca del ecuador) como las de Queensland están en la zona de menor frecuencia relativa por este efecto.

3. Hubo un evento real que redujo cobertura mundial completa, no solo regional. Tras una falla en diciembre de 2021, Sentinel-1B dejó de funcionar, y desde entonces hasta el lanzamiento de reemplazos, solo quedó disponible Sentinel-1A — algunas zonas perdieron cobertura por completo, y muchas otras quedaron con tiempos de revisita más largos. Esto afecta a toda tu serie desde 2022 en adelante, sin importar qué dirección de órbita elijas.

Para tu caso concreto: no hay forma de saber de antemano cuál dirección tiene mejor cobertura en un punto específico sin consultarlo — por eso el chequeo automático que agregamos al script es la forma correcta de manejarlo, en vez de asumir ASCENDING como si fuera universal. Y por el punto 3, aunque elijas la mejor dirección, tu serie 2016-2021 probablemente tenga mejor densidad de datos que 2022 en adelante — otra limitación real que vale la pena mencionar en la metodología de tesis, no solo en el código.

In [6]:
"""
Extrae la serie de humedad de suelo relativa quincenal para un punto,
usando Sentinel-1 GRD (radar SAR) con el método de change detection
(Bauer-Marschallinger et al., 2019 - TU Wien).

MÉTODO: el backscatter de radar (VV, en dB) depende de la humedad del
suelo, pero también de la geometría de observación y la vegetación. El
change detection normaliza esto comparando cada valor contra el rango
histórico observado en ese MISMO punto:

    SM_index = (VV_actual - VV_seco) / (VV_húmedo - VV_seco) * 100

donde VV_seco y VV_húmedo son los percentiles 5 y 95 (no el mínimo/
máximo exacto, para ser robusto a ruido speckle) de la serie histórica.

ESTRATEGIA DE DOBLE ÓRBITA: la cobertura de Sentinel-1 varía mucho por
región y no es uniforme entre las direcciones ascendente/descendente
(depende del plan de observación de ESA, que prioriza distintas zonas
del planeta de forma desigual). En vez de fijar una sola dirección
para toda la serie (lo que puede dejar huecos grandes si esa dirección
tiene poca cobertura en el punto elegido), se calculan referencias
seco/húmedo INDEPENDIENTES para ascendente y descendente, y cada
quincena usa la dirección que tenga imágenes disponibles (si ambas
tienen, se prioriza la de mayor cobertura histórica total). Esto
permite que el script funcione en cualquier punto del planeta sin
tener que adivinar de antemano cuál dirección conviene.

CÓMO LEER EL RESULTADO
-----------------------
sm_index    Índice de humedad de suelo relativa (0-100%). NO es
            humedad volumétrica absoluta (m3/m3) - es una posición
            relativa entre el estado más seco y más húmedo observado
            históricamente en ESE punto, calibrado con la misma
            dirección de órbita que ese dato específico.
orbita      Qué dirección de órbita se usó para esa quincena
            ('ASCENDING', 'DESCENDING', o 'sin_dato' si ninguna de
            las dos tenía imágenes disponibles).
n_imagenes  Cuántas escenas de Sentinel-1 promedia esa quincena.

LIMITACIONES A TENER EN CUENTA:
- Consistencia entre quincenas: como cada quincena puede venir de una
  calibración distinta (ascendente vs descendente), el índice de cada
  quincena es internamente consistente (0-100% relativo a SU propia
  referencia histórica), pero la comparación entre dos quincenas de
  distinta órbita tiene un poco más de incertidumbre que si toda la
  serie viniera de la misma fuente -> revisar la columna 'orbita' antes
  de sacar conclusiones sobre cambios abruptos entre periodos.
- Vegetación densa: en dosel muy cerrado (selva, cultivo maduro), el
  radar "ve" más la vegetación que el suelo, y la relación
  backscatter-humedad se debilita. Es más confiable en suelo desnudo o
  vegetación baja/dispersa.
- Frecuencia: la revisita de Sentinel-1 varía por región y década (se
  degradó desde que S1B dejó de operar en dic-2021) -> algunas
  quincenas pueden seguir sin ninguna imagen en NINGUNA dirección,
  sobre todo en zonas de baja prioridad de observación. Se deja como
  NaN cuando pasa esto - no hay una tercera fuente a la que recurrir.

CITA: Bauer-Marschallinger, B., et al. (2019). Toward global soil
moisture monitoring with Sentinel-1: Harnessing assets and overcoming
obstacles. IEEE TGRS, 57(1), 520-539.
"""
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee

from test_period_utils import build_biweekly_periods


def _get_orbit_reference(base_coll, orbit_pass, point):
    """Calcula VV_seco (p5) y VV_húmedo (p95) históricos para una dirección de órbita."""
    coll = base_coll.filter(ee.Filter.eq('orbitProperties_pass', orbit_pass))
    n_total = coll.size().getInfo()
    if n_total == 0:
        return coll, None, None, 0

    ref_stats = coll.reduce(ee.Reducer.percentile([5, 95])).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=10,
        maxPixels=1e9
    ).getInfo()
    return coll, ref_stats.get('VV_p5'), ref_stats.get('VV_p95'), n_total


def get_soil_moisture_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    base_coll = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(point)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filterDate(str(start), str(end + timedelta(days=1)))
        .select('VV')
    )

    # Referencias independientes para cada dirección de órbita
    coll_asc, vv_seco_asc, vv_humedo_asc, n_asc = _get_orbit_reference(base_coll, 'ASCENDING', point)
    coll_desc, vv_seco_desc, vv_humedo_desc, n_desc = _get_orbit_reference(base_coll, 'DESCENDING', point)
    print(f"[DEBUG] cobertura histórica total: {n_asc} imágenes ASCENDING, {n_desc} imágenes DESCENDING")

    if n_asc == 0 and n_desc == 0:
        raise ValueError("No hay imágenes de Sentinel-1 (VV, IW) en ninguna dirección "
                          "de órbita para este punto y rango de fechas.")

    # Dirección "preferida" cuando ambas tienen datos en una quincena dada:
    # la de mayor cobertura histórica total (referencia más confiable)
    orbita_preferida = 'ASCENDING' if n_asc >= n_desc else 'DESCENDING'

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        n_a = ee.Number(coll_asc.filterDate(p_start, p_end).size()) if n_asc > 0 else ee.Number(0)
        n_d = ee.Number(coll_desc.filterDate(p_start, p_end).size()) if n_desc > 0 else ee.Number(0)

        usar_asc = ee.Algorithms.If(
            orbita_preferida == 'ASCENDING',
            n_a.gt(0),
            n_a.gt(0).And(n_d.eq(0))
        )

        def imagen_para(orbita_coll, vv_seco, vv_humedo, n_img):
            mean_vv = ee.Image(ee.Algorithms.If(
                n_img.gt(0),
                orbita_coll.filterDate(p_start, p_end).mean().rename('VV'),
                ee.Image.constant(0).rename('VV').selfMask()
            ))
            rango = ee.Number(vv_humedo).subtract(vv_seco)
            sm = mean_vv.subtract(vv_seco).divide(rango).multiply(100).max(0).min(100)
            return sm.rename('sm_index')

        if n_asc > 0 and n_desc > 0:
            img_asc = imagen_para(coll_asc, vv_seco_asc, vv_humedo_asc, n_a)
            img_desc = imagen_para(coll_desc, vv_seco_desc, vv_humedo_desc, n_d)
            sm_index_img = ee.Image(ee.Algorithms.If(usar_asc, img_asc, img_desc))
            orbita_usada = ee.Algorithms.If(
                usar_asc, 'ASCENDING',
                ee.Algorithms.If(n_d.gt(0), 'DESCENDING', 'sin_dato')
            )
            n_imagenes = ee.Algorithms.If(usar_asc, n_a, n_d)
        elif n_asc > 0:
            sm_index_img = imagen_para(coll_asc, vv_seco_asc, vv_humedo_asc, n_a)
            orbita_usada = ee.Algorithms.If(n_a.gt(0), 'ASCENDING', 'sin_dato')
            n_imagenes = n_a
        else:
            sm_index_img = imagen_para(coll_desc, vv_seco_desc, vv_humedo_desc, n_d)
            orbita_usada = ee.Algorithms.If(n_d.gt(0), 'DESCENDING', 'sin_dato')
            n_imagenes = n_d

        n_img_band = ee.Image.constant(n_imagenes).toInt().rename('n_imagenes')
        combined = sm_index_img.addBands(n_img_band)

        stats = combined.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=10,
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
            .set('orbita', orbita_usada)
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para todos los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            'lat': lat,
            'lon': lon,
            'sm_index': round(props.get('sm_index'), 2) if props.get('sm_index') is not None else None,
            'orbita': props.get('orbita'),
            'n_imagenes': props.get('n_imagenes'),
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    filas_nulas = df['sm_index'].isna().sum()
    if filas_nulas > 0:
        primeras_nulas = df[df['sm_index'].isna()]['label'].tolist()
        print(f"[AVISO] {filas_nulas} quincena(s) sin ninguna imagen en ninguna "
              f"dirección de órbita: {primeras_nulas}")

    return df


def save_soil_moisture_profile(df, out_prefix="soil_moisture_biweekly", output_dir="../databases"):
    """
    Guarda la serie de humedad de suelo con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Sugarcane_QLD
    LAT = 7.4584221918243045
    LON = -73.222052853104
    
    # Reference points for quick access (commented out):
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    
    df = get_soil_moisture_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_soil_moisture_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-13
[DEBUG] cobertura histórica total: 318 imágenes ASCENDING, 323 imágenes DESCENDING
[AVISO] 11 quincena(s) sin ninguna imagen en ninguna dirección de órbita: ['2016-01_Q1', '2016-05_Q1', '2016-06_Q2', '2016-08_Q1', '2018-08_Q2', '2018-09_Q1', '2018-09_Q2', '2021-12_Q2', '2022-01_Q1', '2022-01_Q2', '2022-02_Q1']
CSV guardado en ../databases/soil_moisture_biweekly-v260813181005.csv (254x8)
  periodo_inicio periodo_fin       label       lat        lon  sm_index  \
0     2016-01-01  2016-01-15  2016-01_Q1  7.458422 -73.222053       NaN   
1     2016-01-16  2016-01-31  2016-01_Q2  7.458422 -73.222053     64.02   
2     2016-02-01  2016-02-15  2016-02_Q1  7.458422 -73.222053     34.74   
3     2016-02-16  2016-02-29  2016-02_Q2  7.458422 -73.222053     91.08   
4     2016-03-01  2016-03-15  2016-03_Q1  7.458422 -73.222053     17.62   
5     2016-03-16  2016-03-31  2016-03_Q2  7.458422 -73.222053     50.64   
6     2016-04-01 